# Singular Value Decomposition Notebook

> Hands-on Build It and Exercises.

## Build It

### Step 1: SVD from scratch using power iteration

The idea: to find the largest singular value and its vectors, use power iteration on A^T A (or A A^T). Then deflate the matrix and repeat for the next singular value.

In [ ]:
```python

import numpy as np

def power_iteration(M, num_iters=100):

    n = M.shape[1]

    v = np.random.randn(n)

    v = v / np.linalg.norm(v)

    for _ in range(num_iters):

        Mv = M @ v

        v = Mv / np.linalg.norm(Mv)

    eigenvalue = v @ M @ v

    return eigenvalue, v

def svd_from_scratch(A, k=None):

    m, n = A.shape

    if k is None:

        k = min(m, n)

    sigmas = []

    us = []

    vs = []

    A_residual = A.copy().astype(float)

    for _ in range(k):

        AtA = A_residual.T @ A_residual

        eigenvalue, v = power_iteration(AtA, num_iters=200)

        if eigenvalue < 1e-10:

            break

        sigma = np.sqrt(eigenvalue)

        u = A_residual @ v / sigma

        sigmas.append(sigma)

        us.append(u)

        vs.append(v)

        A_residual = A_residual - sigma * np.outer(u, v)

    U = np.column_stack(us) if us else np.empty((m, 0))

    S = np.array(sigmas)

    V = np.column_stack(vs) if vs else np.empty((n, 0))

    return U, S, V

In [ ]:
```

### Step 2: Test and compare with NumPy

In [ ]:
```python

np.random.seed(42)

A = np.random.randn(5, 4)

U_ours, S_ours, V_ours = svd_from_scratch(A)

U_np, S_np, Vt_np = np.linalg.svd(A, full_matrices=False)

print("Our singular values:", np.round(S_ours, 4))

print("NumPy singular values:", np.round(S_np, 4))

A_reconstructed = U_ours @ np.diag(S_ours) @ V_ours.T

print(f"Reconstruction error: {np.linalg.norm(A - A_reconstructed):.8f}")

In [ ]:
```

### Step 3: Image compression demo

In [ ]:
```python

def compress_image_svd(image_matrix, k):

    U, S, Vt = np.linalg.svd(image_matrix, full_matrices=False)

    compressed = U[:, :k] @ np.diag(S[:k]) @ Vt[:k, :]

    return compressed

image = np.random.seed(42)

rows, cols = 200, 300

image = np.random.randn(rows, cols)

for k in [1, 5, 10, 20, 50]:

    compressed = compress_image_svd(image, k)

    error = np.linalg.norm(image - compressed) / np.linalg.norm(image)

    original_size = rows * cols

    compressed_size = k * (rows + cols + 1)

    ratio = compressed_size / original_size

    print(f"k={k:>3d}  error={error:.4f}  storage={ratio:.1%}")

In [ ]:
```

### Step 4: Noise reduction

In [ ]:
```python

np.random.seed(42)

clean = np.outer(np.sin(np.linspace(0, 4*np.pi, 100)),

                 np.cos(np.linspace(0, 2*np.pi, 80)))

noise = 0.3 * np.random.randn(100, 80)

noisy = clean + noise

U, S, Vt = np.linalg.svd(noisy, full_matrices=False)

denoised = U[:, :5] @ np.diag(S[:5]) @ Vt[:5, :]

print(f"Noisy error:    {np.linalg.norm(noisy - clean):.4f}")

print(f"Denoised error: {np.linalg.norm(denoised - clean):.4f}")

print(f"Improvement:    {(1 - np.linalg.norm(denoised - clean) / np.linalg.norm(noisy - clean)):.1%}")

In [ ]:
```

### Step 5: Pseudoinverse

In [ ]:
```python

A = np.array([[1, 1], [2, 1], [3, 1]], dtype=float)

b = np.array([3, 5, 6], dtype=float)

U, S, Vt = np.linalg.svd(A, full_matrices=False)

S_inv = np.diag(1.0 / S)

A_pinv = Vt.T @ S_inv @ U.T

x_svd = A_pinv @ b

x_lstsq = np.linalg.lstsq(A, b, rcond=None)[0]

x_pinv = np.linalg.pinv(A) @ b

print(f"SVD pseudoinverse solution:  {x_svd}")

print(f"np.linalg.lstsq solution:   {x_lstsq}")

print(f"np.linalg.pinv solution:    {x_pinv}")

In [ ]:
```

## Exercises

In [ ]:
1. Implement the full SVD from scratch without using power iteration. Instead, compute the eigendecomposition of A^T A to get V and the singular values, then compute U = A V Sigma^{-1}. Compare numerical accuracy with your power iteration version and with NumPy.

2. Load a real grayscale image (or convert one to grayscale). Compress it at ranks 1, 5, 10, 25, 50, 100. For each rank, compute the compression ratio and the relative error. Find the rank where the image becomes visually acceptable.

3. Build a tiny recommendation system. Create a 10x8 user-movie ratings matrix with some known entries. Fill missing entries with row means. Compute SVD and reconstruct a rank-3 approximation. Use the reconstructed matrix to predict the missing ratings. Verify that the predictions are reasonable.

4. Create a 100x50 document-term matrix with 3 synthetic topics. Each topic has 5 associated terms. Add noise. Apply SVD and verify that the top 3 singular values are much larger than the rest. Project documents into the 3D latent space and check that documents from the same topic cluster together.

5. Generate a clean low-rank matrix (rank 3, size 50x40) and add Gaussian noise at different levels (sigma = 0.1, 0.5, 1.0, 2.0). For each noise level, find the optimal truncation rank by sweeping k from 1 to 40 and measuring reconstruction error against the clean matrix. Plot how the optimal k changes with noise level.